In [1]:
# %% [markdown]
# # Diagnostic 4: Probe Classifiers Test
# **Objective:** Fit linear probes on frozen layer representations to measure downstream linear separability.

# %%
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


In [2]:

# 1. Generate synthetic classification problem
X_raw, y_raw = make_classification(n_samples=600, n_features=20, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.3, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)


In [3]:

# 2. Backbone 2-layer network
class BackboneNet(nn.Module):
    def __init__(self, in_features=20, hidden_dim=32, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        h = self.act1(self.fc1(x))
        out = self.fc2(h)
        return out, h

backbone = BackboneNet()


In [4]:

# 3. Extract hidden representations (H) with no grad (frozen backbone)
backbone.eval()
with torch.no_grad():
    _, H_train = backbone(X_train_t)
    _, H_test = backbone(X_test_t)

# 4. Train a lightweight linear probe on H
class LinearProbe(nn.Module):
    def __init__(self, in_dim=32, num_classes=2):
        super().__init__()
        self.linear = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

probe = LinearProbe(in_dim=32, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(probe.parameters(), lr=0.01)

for epoch in range(100):
    optimizer.zero_grad()
    preds = probe(H_train)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()

# Evaluate probe accuracy
with torch.no_grad():
    probe_test_preds = torch.argmax(probe(H_test), dim=1)
    probe_acc = (probe_test_preds == y_test_t).float().mean().item() * 100

print(f"Layer-1 Hidden Representation Probe Accuracy: {probe_acc:.2f}%")
print("\n--- Diagnostic Verdict ---")
print("Linear probe trained successfully on frozen features. Compare probe performance layer-by-layer to isolate representation bottlenecks.")

Layer-1 Hidden Representation Probe Accuracy: 84.44%

--- Diagnostic Verdict ---
Linear probe trained successfully on frozen features. Compare probe performance layer-by-layer to isolate representation bottlenecks.


# Diagnostic 4b: Using Probes to Diagnose Injected Errors

The cell above reports one number (a single layer's probe accuracy) and a verdict that says the probe "trained successfully". A probe accuracy only means something next to **baselines** and **neighbouring layers**. This section adds both, then injects three errors and reads each one off the probe results.

**Reference points used for every model:**
- **Majority-class accuracy:** what a probe scores by guessing the most common class.
- **Raw-input probe:** how linearly separable the data already is before any network.
- **Random-init probe:** the same probe on an untrained copy of the same architecture. Anything the trained model does *not* beat this by is not learned structure.
- **The model's own test accuracy**, compared with the best probe.

**Task:** a 6-class nonlinear problem (a linear probe on the raw input does poorly), so hidden layers have real work to do. The binary task above would hide a narrow bottleneck, because one unit is enough to carry a binary label.

| # | Injected error | What a realistic bug looks like | Expected probe signature |
|---|---|---|---|
| 1 | **Information bottleneck** | Layer-2 width typo (`2` instead of `64`) | Probe accuracy falls at layer 2 and stays low |
| 2 | **Backbone never trained** | Optimizer built from `head.parameters()` only | No lift over the random-init probe at any layer |
| 3 | **Head not loaded** | Checkpoint loaded with `strict=False`, head keys mismatched, head stays random | Probes are high, but the model's own accuracy is near chance |

The verdict rules at the top of the next cell are rules of thumb (drop of 15 points, lift under 5 points, gap over 20 points) and are illustrative, not universal thresholds.

In [5]:
import copy
import numpy as np
from sklearn.preprocessing import StandardScaler

def fit_probe(Htr, ytr, Hte, yte, epochs=200, seed=0):
    """Linear probe on frozen features (reuses LinearProbe from above). Features are standardized with train statistics."""
    torch.manual_seed(seed)
    mu, sd = Htr.mean(0, keepdim=True), Htr.std(0, keepdim=True) + 1e-6
    Htr, Hte = (Htr - mu) / sd, (Hte - mu) / sd
    probe = LinearProbe(Htr.shape[1], int(ytr.max()) + 1)
    opt = optim.Adam(probe.parameters(), lr=0.01)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        crit(probe(Htr), ytr).backward()
        opt.step()
    with torch.no_grad():
        return (probe(Hte).argmax(1) == yte).float().mean().item() * 100

class DeepBackbone(nn.Module):
    """Multi-layer backbone that returns the logits and every hidden representation."""
    def __init__(self, in_features=20, widths=(64, 64, 64), num_classes=6):
        super().__init__()
        dims = [in_features] + list(widths)
        self.layers = nn.ModuleList([nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.head = nn.Linear(dims[-1], num_classes)

    def forward(self, x):
        hs, h = [], x
        for layer in self.layers:
            h = torch.relu(layer(h))
            hs.append(h)
        return self.head(h), hs

def train_model(widths, seed=0, epochs=300, lr=1e-2, train_backbone=True):
    torch.manual_seed(seed)
    m = DeepBackbone(widths=widths)
    params = m.parameters() if train_backbone else m.head.parameters()   # BUG when train_backbone=False
    opt = optim.Adam(params, lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad()
        crit(m(X6_tr)[0], y6_tr).backward()
        opt.step()
    return m

def evaluate(m):
    """Model test accuracy and the linear-probe accuracy at every hidden layer."""
    m.eval()
    with torch.no_grad():
        logits, _ = m(X6_te)
        _, Htr = m(X6_tr)
        _, Hte = m(X6_te)
    acc = (logits.argmax(1) == y6_te).float().mean().item() * 100
    return acc, [fit_probe(a, y6_tr, b, y6_te) for a, b in zip(Htr, Hte)]

_rand_cache = {}
def random_init_probes(widths):
    """Probe accuracy per layer for an UNTRAINED model of the same architecture (seed 123)."""
    if widths not in _rand_cache:
        _rand_cache[widths] = evaluate(train_model(widths, seed=123, epochs=0))[1]
    return _rand_cache[widths]

# Verdict rules of thumb (illustrative thresholds)
DROP_PTS, LIFT_PTS, GAP_PTS = 15.0, 5.0, 20.0

def probe_verdict(raw_probe, probes, rand_probes, model_acc):
    findings = []
    best = raw_probe                                   # best accuracy seen at earlier depth (input counts)
    for k, p in enumerate(probes, start=1):
        if p < best - DROP_PTS:
            findings.append((f"INFO_LOSS@layer{k}", f"probe falls to {p:.1f}% from {best:.1f}% at earlier depth"))
            break
        best = max(best, p)
    lift = max(p - r for p, r in zip(probes, rand_probes))
    if lift < LIFT_PTS:
        findings.append(("NO_LEARNING", f"best lift over random-init is only {lift:+.1f} points"))
    gap = max(probes) - model_acc
    if gap > GAP_PTS:
        findings.append(("HEAD_PROBLEM", f"model accuracy {model_acc:.1f}% is {gap:.1f} points below the best probe ({max(probes):.1f}%)"))
    return findings or [("HEALTHY", "probes rise or hold with depth, beat random-init, and match the model")]

def report(label, model, widths):
    model_acc, probes = evaluate(model)
    rand = random_init_probes(widths)
    findings = probe_verdict(raw_probe, probes, rand, model_acc)
    print(f"=== {label} ===")
    print(f"  {'Representation':<22}{'Probe acc':>10}{'Random-init':>13}{'Lift':>8}")
    print(f"  {'raw input':<22}{raw_probe:>9.1f}%{'':>13}{'':>8}")
    for k, (p, r, w) in enumerate(zip(probes, rand, widths), start=1):
        print(f"  {f'layer {k} ({w} units)':<22}{p:>9.1f}%{r:>12.1f}%{p - r:>+8.1f}")
    print(f"  Model's own test accuracy: {model_acc:.1f}%   (majority-class baseline: {majority:.1f}%)")
    for code, why in findings:
        print(f"  Verdict: {code}: {why}")
    print()
    return dict(label=label, model_acc=model_acc, probes=probes, codes=[c for c, _ in findings])

In [6]:
# What does the original 82.78% actually mean? Compare it with baselines on the ORIGINAL data.
orig_majority = np.bincount(y_test_t.numpy()).max() / len(y_test_t) * 100
orig_raw_probe = fit_probe(X_train_t, y_train_t, X_test_t, y_test_t)
orig_hidden_probe = fit_probe(H_train, y_train_t, H_test, y_test_t)

print(f"Majority-class baseline:                               {orig_majority:.1f}%")
print(f"Probe on RAW INPUT (no network at all):                {orig_raw_probe:.1f}%")
print(f"Probe on the original UNTRAINED layer-1 (this harness): {orig_hidden_probe:.1f}%")
print(f"Probe accuracy reported by the original cell:          {probe_acc:.2f}%")
print()
print("The backbone above was never trained. Its probe score reflects how separable the data already is,")
print("not anything the network learned, and it is not above the raw-input probe.")

Majority-class baseline:                               52.8%
Probe on RAW INPUT (no network at all):                92.2%
Probe on the original UNTRAINED layer-1 (this harness): 81.1%
Probe accuracy reported by the original cell:          84.44%

The backbone above was never trained. Its probe score reflects how separable the data already is,
not anything the network learned, and it is not above the raw-input probe.


In [7]:
# 6-class nonlinear task + reference points shared by all scenarios below
X6, y6 = make_classification(n_samples=3000, n_features=20, n_informative=10, n_redundant=0,
                             n_classes=6, n_clusters_per_class=2, class_sep=1.5, random_state=42)
X6_tr, X6_te, y6_tr, y6_te = train_test_split(X6, y6, test_size=0.3, random_state=42)
scaler = StandardScaler().fit(X6_tr)
X6_tr, X6_te = [torch.tensor(scaler.transform(a), dtype=torch.float32) for a in (X6_tr, X6_te)]
y6_tr, y6_te = torch.tensor(y6_tr), torch.tensor(y6_te)

majority = np.bincount(y6_te.numpy()).max() / len(y6_te) * 100
raw_probe = fit_probe(X6_tr, y6_tr, X6_te, y6_te)
print(f"Majority-class baseline: {majority:.1f}%")
print(f"Probe on raw input:      {raw_probe:.1f}%  (a linear probe cannot solve this task without learned features)")

results = []

Majority-class baseline: 18.3%
Probe on raw input:      65.4%  (a linear probe cannot solve this task without learned features)


In [8]:
# Control: healthy backbone, trained end to end
widths = (64, 64, 64)
healthy = train_model(widths)
results.append(report("Control: healthy backbone", healthy, widths))

=== Control: healthy backbone ===
  Representation         Probe acc  Random-init    Lift
  raw input                  65.4%                     
  layer 1 (64 units)         80.9%        67.2%   +13.7
  layer 2 (64 units)         82.6%        57.6%   +25.0
  layer 3 (64 units)         82.4%        52.0%   +30.4
  Model's own test accuracy: 82.9%   (majority-class baseline: 18.3%)
  Verdict: HEALTHY: probes rise or hold with depth, beat random-init, and match the model



In [9]:
# ERROR 1: information bottleneck from a layer-width typo
widths = (64, 2, 64)                  # typo: layer 2 should be 64 wide
bottleneck = train_model(widths)
results.append(report("Error 1: layer-2 width typo (64, 2, 64)", bottleneck, widths))

=== Error 1: layer-2 width typo (64, 2, 64) ===
  Representation         Probe acc  Random-init    Lift
  raw input                  65.4%                     
  layer 1 (64 units)         79.9%        67.2%   +12.7
  layer 2 (2 units)          59.6%        23.7%   +35.9
  layer 3 (64 units)         63.4%        22.3%   +41.1
  Model's own test accuracy: 63.1%   (majority-class baseline: 18.3%)
  Verdict: INFO_LOSS@layer2: probe falls to 59.6% from 79.9% at earlier depth



In [10]:
# ERROR 2: the backbone never trains (optimizer only receives the head's parameters)
widths = (64, 64, 64)
frozen_backbone = train_model(widths, train_backbone=False)
results.append(report("Error 2: backbone not in optimizer", frozen_backbone, widths))

=== Error 2: backbone not in optimizer ===
  Representation         Probe acc  Random-init    Lift
  raw input                  65.4%                     
  layer 1 (64 units)         66.0%        67.2%    -1.2
  layer 2 (64 units)         59.1%        57.6%    +1.6
  layer 3 (64 units)         54.4%        52.0%    +2.4
  Model's own test accuracy: 51.9%   (majority-class baseline: 18.3%)
  Verdict: NO_LEARNING: best lift over random-init is only +2.4 points



In [11]:
# ERROR 3: the classifier head did not load from the checkpoint (stays at random init)
widths = (64, 64, 64)
broken_head = copy.deepcopy(healthy)  # backbone identical to the healthy control
torch.manual_seed(7)
broken_head.head.reset_parameters()   # simulates strict=False loading with mismatched head keys
results.append(report("Error 3: head left at random init", broken_head, widths))

=== Error 3: head left at random init ===
  Representation         Probe acc  Random-init    Lift
  raw input                  65.4%                     
  layer 1 (64 units)         80.9%        67.2%   +13.7
  layer 2 (64 units)         82.6%        57.6%   +25.0
  layer 3 (64 units)         82.4%        52.0%   +30.4
  Model's own test accuracy: 21.9%   (majority-class baseline: 18.3%)
  Verdict: HEAD_PROBLEM: model accuracy 21.9% is 60.7 points below the best probe (82.6%)



In [12]:
# Summary
print(f"{'Scenario':<42}{'Model acc':>10}   {'Probes: layer1 / layer2 / layer3':<34} Verdict")
print("-" * 112)
for r in results:
    probes = " / ".join(f"{p:5.1f}" for p in r["probes"])
    print(f"{r['label']:<42}{r['model_acc']:>9.1f}%   {probes:<34} {', '.join(r['codes'])}")
print()
print(f"Reference: majority {majority:.1f}%, raw-input probe {raw_probe:.1f}%")

Scenario                                   Model acc   Probes: layer1 / layer2 / layer3   Verdict
----------------------------------------------------------------------------------------------------------------
Control: healthy backbone                      82.9%    80.9 /  82.6 /  82.4              HEALTHY
Error 1: layer-2 width typo (64, 2, 64)        63.1%    79.9 /  59.6 /  63.4              INFO_LOSS@layer2
Error 2: backbone not in optimizer             51.9%    66.0 /  59.1 /  54.4              NO_LEARNING
Error 3: head left at random init              21.9%    80.9 /  82.6 /  82.4              HEAD_PROBLEM

Reference: majority 18.3%, raw-input probe 65.4%


## Takeaways

- **A probe number needs baselines.** In the original cell the backbone was never trained, so its probe score mostly reflects how separable the data already is, and it sits below the raw-input probe. Always compare against the raw input, a random-init model and the majority class.
- **Each error leaves a different signature:**
  - **Error 1 (bottleneck):** probe accuracy is high at layer 1 (79.9%) and falls at the narrow layer (59.6%). Layer 3 is wide again but recovers only a few points (63.4%), because information lost at one layer cannot be restored by later layers. The first drop points to the culprit.
  - **Error 2 (backbone not trained):** the probes match a random-init model at every layer, so there is no lift. The model's own accuracy (51.9%) is well above the majority baseline, so on its own it looks like partial learning, but the probes show the backbone contributes nothing beyond random features.
  - **Error 3 (head not loaded):** the probes are as good as in the healthy control, because the backbone is identical, while the model's own accuracy is near chance. The representation is fine and the fault is downstream, in the head or the loading code.
- **Compare probes with the model's own accuracy.** A large gap in either direction points to different places: a probe far above the model means a head, loading or evaluation problem, and a model that matches a weak probe means the features are the limit.
- **Limits:** a probe shows information is *present*, not that the model *uses* it. Checking for shortcuts or confounders needs control tasks (probes on random labels) and ablations, which this notebook does not demonstrate. Linear probes on small data also vary by a few points between runs, so look for large gaps, not small ones.